# 05 — Supervised Fine-Tuning
Shows the loss mask side-by-side with tokens. This is the differentiator.

In [1]:
import torch
import tiktoken
from sft.data import format_conversation, build_batch

enc = tiktoken.get_encoding("gpt2")

## Visualise the loss mask — the most important concept in SFT

In [2]:
convo = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is 2+2?"},
    {"role": "assistant", "content": "It is 4."},
]

ids, mask = build_batch([convo], enc, block_size=128)
ids, mask = ids[0].tolist(), mask[0].tolist()

print(f"{'Token':>12}  {'ID':>6}  {'Mask':>6}  {'Trains?'}")
print("-" * 45)
for i, (tok_id, m) in enumerate(zip(ids[:60], mask[:60])):
    if tok_id == 0: break
    tok = repr(enc.decode([tok_id]))
    trains = "YES ←" if m == 1.0 else ""
    print(f"{tok:>12}  {tok_id:>6}  {m:>6.1f}  {trains}")

       Token      ID    Mask  Trains?
---------------------------------------------
         '<'      27     0.0  
         '|'      91     0.0  
        'im'     320     0.0  
         '_'      62     0.0  
     'start'    9688     0.0  
         '|'      91     0.0  
         '>'      29     0.0  
    'system'   10057     0.0  
        '\n'     198     0.0  
       'You'    1639     0.0  
      ' are'     389     0.0  
        ' a'     257     0.0  
  ' helpful'    7613     0.0  
' assistant'    8796     0.0  
         '.'      13     0.0  
         '<'      27     0.0  
         '|'      91     0.0  
        'im'     320     0.0  
         '_'      62     0.0  
       'end'     437     0.0  
         '|'      91     0.0  
         '>'      29     0.0  
        '\n'     198     0.0  
         '<'      27     0.0  
         '|'      91     0.0  
        'im'     320     0.0  
         '_'      62     0.0  
     'start'    9688     0.0  
         '|'      91     0.0  
         '>'     

## Before vs after masking — loss comparison

In [3]:
from model.config import GPT2_CONFIG
from model.gpt import GPT
from sft.trainer import sft_loss

try:
    ckpt  = torch.load("../checkpoints/sft.pt", map_location="cpu", weights_only=False)
    model = GPT(GPT2_CONFIG)
    model.load_state_dict(ckpt["model"])
    model.eval()

    input_ids, loss_mask = build_batch([convo], enc, block_size=GPT2_CONFIG.block_size)
    full_mask   = torch.ones_like(loss_mask)
    loss_masked = sft_loss(model, input_ids, loss_mask)
    loss_full   = sft_loss(model, input_ids, full_mask)
    print(f"\nLoss (assistant tokens only): {loss_masked.item():.4f}")
    print(f"Loss (all tokens):            {loss_full.item():.4f}")
except FileNotFoundError:
    print("Run sft/trainer.py first to generate checkpoints/sft.pt")

Run sft/trainer.py first to generate checkpoints/sft.pt
